# 🦁 Conser-vision — Wildlife Classification · Google Colab Training

**Competition:** [DrivenData #87](https://www.drivendata.org/competitions/87/)  
**Task:** Multi-class image classification · 8 wildlife classes  
**Metric:** Log-loss (lower is better)

---
## Before you start
1. `Runtime → Change runtime type → T4 GPU`
2. Make sure Google Drive contains the CSV files under `MyDrive/ML_projects/Conser_vision/data/raw/`:
   ```
   MyDrive/ML_projects/Conser_vision/data/raw/
   ├── train_features.csv
   ├── train_labels.csv
   ├── test_features.csv
   └── train_features.zip   ← zipped image folder
   ```
3. Run all cells top-to-bottom with `Runtime → Run all`


## 1 · Environment Setup


In [ ]:
# ── GPU check ──────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU"
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
print(f"  torch {torch.__version__}  |  CUDA {torch.version.cuda}")


✓ GPU: Tesla T4  |  VRAM: 15.6 GB
  torch 2.10.0+cu128  |  CUDA 12.8


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURE THESE PATHS — edit only here                     ║
# ╚══════════════════════════════════════════════════════════════╝
DRIVE_BASE = Path("/content/drive/MyDrive/ML_projects/Conser_vision")
DRIVE_DATA = DRIVE_BASE / "data/raw"
DRIVE_CKPT = DRIVE_BASE / "checkpoints"
DRIVE_SUB  = DRIVE_BASE / "submissions"
# ╚══════════════════════════════════════════════════════════════╝

PROJECT_ROOT = Path("/content/conser-vision")
LOCAL_DATA   = Path("/content/data_local")

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_SUB]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive base  : {DRIVE_BASE}")
print(f"Drive data  : {DRIVE_DATA}")
print(f"Drive ckpt  : {DRIVE_CKPT}")


Mounted at /content/drive
Drive base  : /content/drive/MyDrive/ML_projects/Conser_vision
Drive data  : /content/drive/MyDrive/ML_projects/Conser_vision/data/raw
Drive ckpt  : /content/drive/MyDrive/ML_projects/Conser_vision/checkpoints


In [ ]:
# ── Install packages ───────────────────────────────────────────
# Colab already has torch, torchvision, numpy, pandas, sklearn, PIL
!pip install -q timm==1.0.3 albumentations==1.4.7 pyyaml tqdm
print("✓ Packages installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 13.5 MB/s eta 0:00:00
✓ Packages installed


## 2 · Project Source Files *(run once — collapse after)*


In [ ]:
# ── Create directory structure ─────────────────────────────────
from pathlib import Path

PROJECT_ROOT = Path("/content/conser-vision")

for d in [
    "src/data", "src/models", "src/training", "src/evaluation",
    "utils", "configs", "data/raw", "data/processed",
    "models/weights", "submissions",
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

for pkg in ["src", "src/data", "src/models", "src/training", "src/evaluation", "utils"]:
    init = PROJECT_ROOT / pkg / "__init__.py"
    if not init.exists():
        init.write_text("")

print("✓ Directory structure created at", PROJECT_ROOT)


✓ Directory structure created at /content/conser-vision


In [ ]:
%%writefile /content/conser-vision/utils/seed.py
"""
utils/seed.py
-------------
Global seed setter for reproducibility.
"""
from __future__ import annotations
import os
import random
import numpy as np


def set_global_seed(seed: int = 42) -> None:
    """Set random seeds for Python, NumPy, PyTorch (and TF if available)."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass


Writing /content/conser-vision/utils/seed.py


In [ ]:
%%writefile /content/conser-vision/src/data/dataset.py
"""
src/data/dataset.py
--------------------
PyTorch Dataset for camera-trap image classification.
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


CLASS_NAMES = [
    "antelope_duiker", "bird", "blank", "civet_genet",
    "hog", "leopard", "monkey_prosimian", "rodent",
]
CLASS_TO_IDX: dict[str, int] = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS: dict[int, str] = {i: c for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)


class WildlifeDataset(Dataset):
    """Camera-trap image dataset for Conser-vision competition."""

    def __init__(
        self,
        df: pd.DataFrame,
        images_dir: str | Path,
        transform: Optional[Callable] = None,
        is_test: bool = False,
        id_col: str = "id",
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.is_test = is_test
        self.id_col = id_col
        if not is_test:
            self.labels: np.ndarray = self.df[CLASS_NAMES].values.astype(np.float32)
        self.image_ids: list[str] = self.df[id_col].tolist()

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, idx: int) -> Image.Image:
        image_id = self.image_ids[idx]
        img_path = self.images_dir / f"{image_id}.jpg"
        if not img_path.exists():
            for ext in [".JPG", ".jpeg", ".JPEG", ".png", ".PNG"]:
                alt = self.images_dir / f"{image_id}{ext}"
                if alt.exists():
                    img_path = alt
                    break
        return Image.open(img_path).convert("RGB")

    def __getitem__(self, idx: int) -> dict:
        img = self._load_image(idx)
        if self.transform is not None:
            if type(self.transform).__module__.startswith("albumentations"):
                img_tensor: torch.Tensor = self.transform(image=np.array(img))["image"]
            else:
                img_tensor = self.transform(img)
        else:
            img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        sample: dict = {"image": img_tensor, "id": self.image_ids[idx]}
        if not self.is_test:
            sample["label"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return sample


def load_dataframes(
    train_features_path: str,
    train_labels_path: str,
    test_features_path: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and merge train/test DataFrames."""
    train_features = pd.read_csv(train_features_path)
    train_labels   = pd.read_csv(train_labels_path)
    test_df        = pd.read_csv(test_features_path)
    train_df = train_features.merge(train_labels, on="id", how="left")
    print(f"Train: {len(train_df):,} samples")
    print(f"Test:  {len(test_df):,} samples")
    print(f"Class distribution:\n{train_labels[CLASS_NAMES].sum().to_string()}")
    return train_df, test_df


Writing /content/conser-vision/src/data/dataset.py


In [ ]:
%%writefile /content/conser-vision/src/data/transforms.py
"""
src/data/transforms.py
-----------------------
Augmentation pipelines using torchvision.transforms.v2.
"""
from __future__ import annotations

import torch
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_train_transforms(image_size: int = 224) -> T.Compose:
    """Strong augmentation for training (camera-trap specific)."""
    return T.Compose([
        T.RandomResizedCrop(image_size, scale=(0.6, 1.0), ratio=(0.75, 1.33),
                            interpolation=InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomRotation(degrees=15, interpolation=InterpolationMode.BICUBIC),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        T.RandomGrayscale(p=0.1),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.2),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_val_transforms(image_size: int = 224) -> T.Compose:
    """Deterministic pipeline for validation and test."""
    resize_size = int(image_size * (256 / 224))
    return T.Compose([
        T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
        T.CenterCrop(image_size),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_tta_transforms(image_size: int = 224) -> list[T.Compose]:
    resize_size = int(image_size * (256 / 224))
    norm = [
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]

    transforms = []
    crops = [
        T.Compose([T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
                   T.CenterCrop(image_size), *norm]),                          # center
        T.Compose([T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
                   T.CenterCrop(image_size), T.RandomHorizontalFlip(p=1.0), *norm]),  # hflip
        T.Compose([T.Resize(image_size, interpolation=InterpolationMode.BICUBIC),
                   T.CenterCrop(image_size), *norm]),                          # smaller scale
        T.Compose([T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
                   T.RandomCrop(image_size), *norm]),                          # random crop A
        T.Compose([T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
                   T.RandomCrop(image_size), *norm]),                          # random crop B
    ]
    return crops


Writing /content/conser-vision/src/data/transforms.py


In [ ]:
%%writefile /content/conser-vision/src/models/model.py
"""
src/models/model.py
--------------------
Image classifier built on top of timm pretrained backbones.
"""
from __future__ import annotations

import torch
import torch.nn as nn
from timm.models import create_model


class WildlifeClassifier(nn.Module):
    """Wildlife species classifier with a timm backbone."""

    def __init__(self, model_name: str = "efficientnet_b3", num_classes: int = 8,
                 pretrained: bool = True, dropout: float = 0.3) -> None:
        super().__init__()
        self.model_name  = model_name
        self.num_classes = num_classes
        self.backbone = create_model(model_name, pretrained=pretrained,
                                     num_classes=0, global_pool="avg")
        num_features: int = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Dropout(p=dropout),
            nn.Linear(num_features, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

    def get_optimizer_param_groups(self, lr: float = 1e-4,
                                   lr_backbone_multiplier: float = 0.1) -> list[dict]:
        return [
            {"params": self.backbone.parameters(), "lr": lr * lr_backbone_multiplier, "name": "backbone"},
            {"params": self.head.parameters(),     "lr": lr,                          "name": "head"},
        ]


def build_model(model_name: str, num_classes: int = 8, pretrained: bool = True,
                dropout: float = 0.3, checkpoint_path: str | None = None) -> WildlifeClassifier:
    """Instantiate a WildlifeClassifier, optionally loading from checkpoint."""
    model = WildlifeClassifier(model_name=model_name, num_classes=num_classes,
                               pretrained=pretrained, dropout=dropout)
    if checkpoint_path is not None:
        state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        model.load_state_dict(state)
        print(f"Loaded checkpoint from {checkpoint_path}")
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model: {model_name}  |  Params: {n_params:.1f}M  |  Classes: {num_classes}")
    return model


Writing /content/conser-vision/src/models/model.py


In [ ]:
%%writefile /content/conser-vision/src/training/train.py
"""
src/training/train.py
----------------------
Training loop with StratifiedKFold CV, mixed-precision, early stopping,
and mid-fold resume support for Colab interruptions.
"""
from __future__ import annotations

import time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.data.dataset import WildlifeDataset, CLASS_NAMES, NUM_CLASSES
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.model import build_model
from utils.seed import set_global_seed


def _save_full_checkpoint(
    path: Path,
    fold: int,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: GradScaler,
    best_log_loss: float,
    patience_counter: int,
    best_oof: Optional[np.ndarray],
) -> None:
    """Save a full training checkpoint for mid-fold resume."""
    torch.save({
        "fold": fold,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_log_loss": best_log_loss,
        "patience_counter": patience_counter,
        "best_oof": best_oof,
    }, path)


def _load_full_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: GradScaler,
) -> dict:
    """Load a full training checkpoint and restore all states in-place.

    Returns:
        Dict with 'epoch', 'best_log_loss', 'patience_counter', 'best_oof'.
    """
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    print(f"  Resumed from epoch {ckpt['epoch'] + 1} "
          f"(best log-loss so far: {ckpt['best_log_loss']:.4f})")
    return {
        "epoch": ckpt["epoch"],
        "best_log_loss": ckpt["best_log_loss"],
        "patience_counter": ckpt["patience_counter"],
        "best_oof": ckpt["best_oof"],
    }


def train_one_epoch(model, loader, optimizer, criterion, scaler, device,
                    gradient_clip=1.0):
    model.train()
    total_loss = 0.0
    n_samples  = 0
    for batch in tqdm(loader, desc="  Train", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(images)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        n_samples  += images.size(0)
    return {"loss": total_loss / n_samples}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_samples  = 0
    all_probs: list[np.ndarray]  = []
    all_labels: list[np.ndarray] = []
    for batch in tqdm(loader, desc="  Val", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        with autocast("cuda"):
            logits = model(images)
            loss   = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy())
        total_loss += loss.item() * images.size(0)
        n_samples  += images.size(0)
    all_probs_np  = np.concatenate(all_probs)
    all_labels_np = np.concatenate(all_labels)
    true_classes  = all_labels_np.argmax(axis=1)
    ll = log_loss(true_classes, all_probs_np, labels=list(range(NUM_CLASSES)))
    return {"loss": total_loss / n_samples, "log_loss": ll,
            "probs": all_probs_np, "labels": all_labels_np}


def train_fold(fold: int, train_df: pd.DataFrame, val_df: pd.DataFrame,
               images_dir: str, cfg: dict, device: torch.device,
               output_dir: Path, resume_dir: Optional[Path] = None):
    """Train a single fold with optional mid-fold resume.

    Args:
        fold: Zero-based fold index.
        train_df: Training split DataFrame.
        val_df: Validation split DataFrame.
        images_dir: Path to training images.
        cfg: Full config dict.
        device: torch device.
        output_dir: Directory for best-model checkpoints.
        resume_dir: Directory to look for resume checkpoints (e.g. Google Drive).
                    If None, no resume is attempted.

    Returns:
        Tuple of (best_val_log_loss, oof_predictions).
    """
    print(f"\n{'='*60}")
    print(f"  FOLD {fold+1}")
    print(f"{'='*60}")
    set_global_seed(cfg["general"]["seed"] + fold)
    model_cfg = cfg["baseline"]

    train_ds = WildlifeDataset(train_df, images_dir,
                               transform=get_train_transforms(model_cfg["image_size"]))
    val_ds   = WildlifeDataset(val_df,   images_dir,
                               transform=get_val_transforms(model_cfg["image_size"]))
    nw = 2 if device.type == "cuda" else 0
    train_loader = DataLoader(train_ds, batch_size=model_cfg["batch_size"],
                              shuffle=True,  num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=model_cfg["batch_size"] * 2,
                              shuffle=False, num_workers=nw, pin_memory=True)

    model = build_model(
        model_name=model_cfg["model_name"],
        num_classes=cfg["general"]["num_classes"],
        pretrained=model_cfg["pretrained"],
        dropout=model_cfg["dropout"],
    ).to(device)

    param_groups = model.get_optimizer_param_groups(
        lr=model_cfg["learning_rate"], lr_backbone_multiplier=0.1)
    optimizer = AdamW(param_groups, weight_decay=model_cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=model_cfg["num_epochs"], eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=model_cfg["label_smoothing"])
    scaler    = GradScaler("cuda", enabled=model_cfg["mixed_precision"] and device.type == "cuda")

    best_log_loss = float("inf")
    best_oof: Optional[np.ndarray] = None
    patience_counter = 0
    patience = model_cfg["early_stopping_patience"]
    start_epoch = 0

    # ── Attempt mid-fold resume ───────────────────────────────
    if resume_dir is not None:
        resume_ckpt = Path(resume_dir) / f"fold{fold+1}_resume.pth"
        if resume_ckpt.exists():
            state = _load_full_checkpoint(resume_ckpt, model, optimizer,
                                          scheduler, scaler)
            start_epoch = state["epoch"] + 1
            best_log_loss = state["best_log_loss"]
            patience_counter = state["patience_counter"]
            best_oof = state["best_oof"]

    for epoch in range(start_epoch, model_cfg["num_epochs"]):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, optimizer, criterion,
                                  scaler, device, model_cfg["gradient_clip"])
        val_m   = validate(model, val_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - t0
        print(f"  Ep {epoch+1:3d}/{model_cfg['num_epochs']} | "
              f"TrLoss {train_m['loss']:.4f} | "
              f"ValLoss {val_m['loss']:.4f} | "
              f"LogLoss {val_m['log_loss']:.4f} | "
              f"LR {scheduler.get_last_lr()[0]:.2e} | "
              f"{elapsed:.0f}s")
        if val_m["log_loss"] < best_log_loss:
            best_log_loss = val_m["log_loss"]
            best_oof = val_m["probs"]
            patience_counter = 0
            ckpt = output_dir / f"fold{fold+1}_best.pth"
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "val_log_loss": best_log_loss, "fold": fold}, ckpt)
            print(f"  ✓ Best checkpoint saved → {ckpt}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        # ── Save resume checkpoint every epoch ────────────────
        if resume_dir is not None:
            resume_path = Path(resume_dir) / f"fold{fold+1}_resume.pth"
            _save_full_checkpoint(
                path=resume_path, fold=fold, epoch=epoch, model=model,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                best_log_loss=best_log_loss, patience_counter=patience_counter,
                best_oof=best_oof,
            )

    # ── Clean up resume checkpoint after fold completes ───────
    if resume_dir is not None:
        resume_path = Path(resume_dir) / f"fold{fold+1}_resume.pth"
        if resume_path.exists():
            resume_path.unlink()
            print(f"  Cleaned up resume checkpoint: {resume_path.name}")

    print(f"\n  Fold {fold+1} best log-loss: {best_log_loss:.4f}")
    return best_log_loss, best_oof


def run_cv(train_df: pd.DataFrame, images_dir: str, cfg: dict,
           device: torch.device, output_dir: Path,
           resume_dir: Optional[Path] = None):
    """Full StratifiedKFold cross-validation with resume support.

    Args:
        train_df: Full training DataFrame.
        images_dir: Path to training images.
        cfg: Full config dict.
        device: torch device.
        output_dir: Directory for best-model checkpoints.
        resume_dir: Directory for resume checkpoints (e.g. DRIVE_CKPT).
                    Completed folds (those with a fold*_best.pth but no
                    fold*_resume.pth) are skipped automatically.

    Returns:
        Tuple of (oof_predictions, fold_scores).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    n_splits = cfg["cross_validation"]["n_splits"]

    # --- Site-aware CV split ---
    # The public test set uses camera-trap sites that are COMPLETELY DISJOINT
    # from the train set (verified in reports/diagnostic_audit.md). The local
    # val fold MUST therefore also be site-disjoint, otherwise the metric
    # measures in-site memorisation instead of out-of-site generalisation.
    if "site" not in train_df.columns:
        raise KeyError("train_df must contain a 'site' column for site-aware CV.")
    y = train_df[CLASS_NAMES].values.argmax(axis=1)
    groups = train_df["site"].values
    target_series = train_df[CLASS_NAMES].idxmax(axis=1)  # kept for back-compat

    oof_preds   = np.zeros((len(train_df), cfg["general"]["num_classes"]), dtype=np.float32)
    fold_scores: list[float] = []
    all_val_idx = np.array([], dtype=np.intp)

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=cfg["cross_validation"]["shuffle"],
        random_state=cfg["general"]["seed"],
    )
    all_splits = list(sgkf.split(np.arange(len(train_df)), y, groups))
    if n_splits == 1:
        splits = [all_splits[0]]
    elif n_splits == 5:
        splits = all_splits
    else:
        raise ValueError(
            f"cross_validation.n_splits must be 1 or 5, got {n_splits}. "
            "Site-aware CV is built from a fixed 5-way StratifiedGroupKFold."
        )

    for fold, (train_idx, val_idx) in enumerate(splits):
        # --- Guardrail: assert no site leakage between train and val folds ---
        train_groups = set(groups[train_idx])
        val_groups = set(groups[val_idx])
        leaked = train_groups & val_groups
        assert len(leaked) == 0, (
            f"GROUP LEAKAGE in fold {fold}: "
            f"{len(leaked)} sites shared between train and val folds"
        )
        val_class_counts = np.bincount(y[val_idx], minlength=cfg["general"]["num_classes"])
        val_class_pct = val_class_counts / max(len(val_idx), 1) * 100
        print(
            f"[fold {fold}] "
            f"train rows={len(train_idx):>6}  val rows={len(val_idx):>6} | "
            f"train sites={len(train_groups):>4}  val sites={len(val_groups):>4}  leaked=0",
            flush=True,
        )
        print(
            "           val class dist: "
            + ", ".join(f"{c}={p:.1f}%" for c, p in zip(CLASS_NAMES, val_class_pct)),
            flush=True,
        )

        local_best = output_dir / f"fold{fold+1}_best.pth"
        drive_best = (Path(resume_dir) / f"fold{fold+1}_best.pth") if resume_dir is not None else None

        best_ckpt_exists = local_best.exists() or (drive_best is not None and drive_best.exists())
        resume_ckpt_exists = (
            resume_dir is not None
            and (Path(resume_dir) / f"fold{fold+1}_resume.pth").exists()
        )
        best_ckpt = local_best if local_best.exists() else drive_best

        print(f"[DEBUG fold{fold+1}] local_best exists: {local_best.exists()}")
        print(f"[DEBUG fold{fold+1}] drive_best exists: {drive_best.exists() if drive_best else None}")
        print(f"[DEBUG fold{fold+1}] best_ckpt_exists: {best_ckpt_exists}")
        print(f"[DEBUG fold{fold+1}] resume_ckpt_exists: {resume_ckpt_exists}")

        if best_ckpt_exists and not resume_ckpt_exists:
            ckpt_data = torch.load(best_ckpt, map_location="cpu", weights_only=False)
            fold_ll = ckpt_data["val_log_loss"]
            print(f"\n{'='*60}")
            print(f"  FOLD {fold+1} — SKIPPED (already completed, "
                  f"log-loss: {fold_ll:.4f})")
            print(f"{'='*60}")

            model_cfg = cfg["baseline"]
            model = build_model(
                model_name=model_cfg["model_name"],
                num_classes=cfg["general"]["num_classes"],
                pretrained=False, dropout=model_cfg["dropout"],
                checkpoint_path=str(best_ckpt),
            ).to(device)
            val_ds = WildlifeDataset(
                train_df.iloc[val_idx], images_dir,
                transform=get_val_transforms(model_cfg["image_size"]))
            val_loader = DataLoader(val_ds, batch_size=model_cfg["batch_size"] * 2,
                                    shuffle=False, num_workers=2, pin_memory=True)
            criterion = nn.CrossEntropyLoss(
                label_smoothing=model_cfg["label_smoothing"])
            val_m = validate(model, val_loader, criterion, device)
            fold_oof = val_m["probs"]
            del model
            torch.cuda.empty_cache()
        else:
            fold_ll, fold_oof = train_fold(
                fold=fold, train_df=train_df.iloc[train_idx],
                val_df=train_df.iloc[val_idx],
                images_dir=images_dir, cfg=cfg, device=device,
                output_dir=output_dir, resume_dir=resume_dir)

            if resume_dir is not None:
                import shutil
                best_src = output_dir / f"fold{fold+1}_best.pth"
                if best_src.exists():
                    shutil.copy2(best_src, Path(resume_dir) / best_src.name)

        oof_preds[val_idx] = fold_oof
        fold_scores.append(fold_ll)
        all_val_idx = np.concatenate([all_val_idx, val_idx])

    overall_ll = log_loss(
        train_df[CLASS_NAMES].values.argmax(axis=1)[all_val_idx],
        oof_preds[all_val_idx], labels=list(range(cfg["general"]["num_classes"])))

    print(f"\n{'='*60}")
    print(f"  CV Results:")
    for i, s in enumerate(fold_scores):
        print(f"    Fold {i+1}: {s:.4f}")
    print(f"  Mean fold log-loss : {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")
    print(f"  OOF  log-loss      : {overall_ll:.4f}")
    print(f"{'='*60}")

    oof_df = train_df[["id"]].copy()
    for i, cls in enumerate(CLASS_NAMES):
        oof_df[cls] = oof_preds[:, i]
    oof_path = Path(cfg["paths"]["oof_dir"]) / "oof_predictions.csv"
    oof_path.parent.mkdir(parents=True, exist_ok=True)
    oof_df.to_csv(oof_path, index=False)
    print(f"  OOF saved → {oof_path}")
    return oof_preds, fold_scores


In [ ]:
%%writefile /content/conser-vision/src/evaluation/predict.py
"""
src/evaluation/predict.py
--------------------------
Generate test set predictions using trained fold checkpoints.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.data.dataset import WildlifeDataset, CLASS_NAMES
from src.data.transforms import get_val_transforms, get_tta_transforms
from src.models.model import build_model


@torch.no_grad()
def predict_single_checkpoint(model, loader, device) -> np.ndarray:
    """Get softmax probabilities from a single model checkpoint."""
    model.eval()
    all_probs: list[np.ndarray] = []
    for batch in tqdm(loader, desc="  Predicting", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        probs  = F.softmax(model(images), dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs)


@torch.no_grad()
def predict_with_tta(model, test_df, images_dir, image_size, device,
                     batch_size=32) -> np.ndarray:
    """Predict with TTA: average over multiple views."""
    tta_transforms = get_tta_transforms(image_size)
    tta_probs: list[np.ndarray] = []
    for i, transform in enumerate(tta_transforms):
        print(f"  TTA view {i+1}/{len(tta_transforms)}")
        ds = WildlifeDataset(test_df, images_dir, transform=transform, is_test=True)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
        tta_probs.append(predict_single_checkpoint(model, loader, device))
    return np.mean(tta_probs, axis=0)


def generate_submission(test_df, images_dir, checkpoint_dir, cfg, device,
                        use_tta=True, output_path=None, folds=None):
    """Average predictions across all fold checkpoints."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoints    = sorted(checkpoint_dir.glob("fold*_best.pth"))

    if folds is not None:
        checkpoints = [
            c for c in checkpoints
            if any(c.name.startswith(f"fold{f}_") for f in folds)
        ]

    print(f"Found {len(checkpoints)} checkpoints: {[c.name for c in checkpoints]}")

    model_cfg = cfg["baseline"]
    all_fold_probs: list[np.ndarray] = []

    for ckpt_path in checkpoints:
        print(f"\nLoading {ckpt_path.name}")
        model = build_model(
            model_name=model_cfg["model_name"],
            num_classes=cfg["general"]["num_classes"],
            pretrained=False, dropout=model_cfg["dropout"],
            checkpoint_path=str(ckpt_path)).to(device)
        if use_tta:
            probs = predict_with_tta(model, test_df, images_dir,
                                     image_size=model_cfg["image_size"],
                                     device=device, batch_size=model_cfg["batch_size"])
        else:
            ds     = WildlifeDataset(test_df, images_dir, is_test=True,
                                     transform=get_val_transforms(model_cfg["image_size"]))
            loader = DataLoader(ds, batch_size=model_cfg["batch_size"] * 2,
                                shuffle=False, num_workers=2)
            probs = predict_single_checkpoint(model, loader, device)
        all_fold_probs.append(probs)
        del model; torch.cuda.empty_cache()

    final_probs = np.mean(all_fold_probs, axis=0)
    sub_df = pd.DataFrame({"id": test_df["id"].values})
    for i, cls in enumerate(CLASS_NAMES):
        sub_df[cls] = final_probs[:, i]

    assert (sub_df[CLASS_NAMES].sum(axis=1) - 1.0).abs().max() < 1e-4

    if output_path is not None:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        sub_df.to_csv(output_path, index=False)
        print(f"\n✓ Submission saved → {output_path}")
        print(sub_df.head())
    return sub_df


Writing /content/conser-vision/src/evaluation/predict.py


In [ ]:
%%writefile /content/conser-vision/configs/config.yaml
# Conser-vision — Colab config

general:
  seed: 42
  num_classes: 8
  class_names:
    - antelope_duiker
    - bird
    - blank
    - civet_genet
    - hog
    - leopard
    - monkey_prosimian
    - rodent

data:
  train_features_path: data/raw/train_features.csv
  train_labels_path:   data/raw/train_labels.csv
  test_features_path:  data/raw/test_features.csv
  train_images_dir:    data/raw/train_features
  test_images_dir:     data/raw/test_features
  id_col: id

cross_validation:
  # Site-aware CV: StratifiedGroupKFold(n_splits=5) on the `site` column.
  strategy: StratifiedGroupKFold
  group_col: site
  n_splits: 5
  shuffle: true

baseline:
  model_name: efficientnet_b3
  pretrained: true
  dropout: 0.3
  image_size: 224
  batch_size: 32
  num_epochs: 30
  learning_rate: 1.0e-4
  weight_decay:  1.0e-4
  label_smoothing: 0.1
  early_stopping_patience: 7
  mixed_precision: true
  gradient_clip: 1.0

tta:
  enabled: true
  n_augments: 5

paths:
  model_dir:      models/weights
  checkpoint_dir: models/weights
  submission_dir: submissions
  oof_dir:        data/processed



## 3 · Data Setup

**Option A (recommended):** upload `train_features.zip` directly from your PC — fastest, bypasses Drive latency.  
**Option B:** use the zip already on Drive under `DRIVE_DATA/train_features.zip`.

Run the cell below — it will ask which option you want.


In [ ]:
# ── Load data into Colab RAM ───────────────────────────────────
# Option A: upload zip directly from your PC (fastest)
# Option B: copy zip from Drive (no upload needed)
#
# Set USE_DRIVE_ZIP = True to use the zip already on Drive,
# or False to trigger a file upload dialog.
USE_DRIVE_ZIP = True  # ← change to False to upload from PC

import os
import shutil
import zipfile
from pathlib import Path

LOCAL_DATA = Path("/content/data_local")

if LOCAL_DATA.exists():
    print("✓ Data already in RAM — skipping.")
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)

    # ── Step 1: get the zip ────────────────────────────────────────
    if USE_DRIVE_ZIP:
        zip_src = DRIVE_DATA / "train_features.zip"
        if not zip_src.exists():
            raise FileNotFoundError(f"Zip not found on Drive: {zip_src}")
        zip_local = Path("/content/train_features.zip")
        print(f"Copiando zip da Drive → /content/ ...")
        shutil.copy2(str(zip_src), str(zip_local))
        print("  ✓ Zip copiato")
    else:
        from google.colab import files
        print("Seleziona train_features.zip dal tuo PC:")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        zip_local = Path(f"/content/{zip_name}")

    # ── Step 2: extract images ─────────────────────────────────────
    print(f"Estraendo {zip_local.name} → {LOCAL_DATA} ...")
    with zipfile.ZipFile(str(zip_local), "r") as zf:
        zf.extractall(str(LOCAL_DATA))
    zip_local.unlink()  # free space
    print("  ✓ Immagini estratte")

    # ── Step 3: copy CSV files from Drive ─────────────────────────
    for csv_file in ["train_features.csv", "train_labels.csv", "test_features.csv"]:
        src = DRIVE_DATA / csv_file
        if not src.exists():
            raise FileNotFoundError(f"CSV non trovato su Drive: {src}")
        shutil.copy2(str(src), str(LOCAL_DATA / csv_file))
        print(f"  ✓ {csv_file}")

    # ── Step 4: verify image dir (handle nested zip structure) ─────
    img_dir = LOCAL_DATA / "train_features"
    if not img_dir.exists():
        nested = img_dir / "train_features"
        if nested.exists():
            nested.rename(img_dir)
            print("  ✓ Corretta struttura zip annidata")
        else:
            print(f"  ⚠ Cartella train_features non trovata in {LOCAL_DATA}")
            print(f"    Contenuto: {list(LOCAL_DATA.iterdir())}")

    n_imgs = len(list((LOCAL_DATA / 'train_features').glob('*.jpg')))
    print(f"\n✓ Data pronto in RAM: {n_imgs:,} immagini train")


Copiando zip da Drive → /content/ ...
  ✓ Zip copiato
Estraendo train_features.zip → /content/data_local ...
  ✓ Immagini estratte
  ✓ train_features.csv
  ✓ train_labels.csv
  ✓ test_features.csv

✓ Data pronto in RAM: 16,488 immagini train


In [ ]:
# ── Patch config con path locali + verifica ────────────────────
import os
import sys
import yaml
import pandas as pd

os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["train_features_path"] = str(LOCAL_DATA / "train_features.csv")
cfg["data"]["train_labels_path"]   = str(LOCAL_DATA / "train_labels.csv")
cfg["data"]["test_features_path"]  = str(LOCAL_DATA / "test_features.csv")
cfg["data"]["train_images_dir"]    = str(LOCAL_DATA / "train_features")
cfg["data"]["test_images_dir"]     = str(LOCAL_DATA / "test_features")

with open("configs/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

# Verifica
train_df = pd.read_csv(cfg["data"]["train_features_path"])
labels   = pd.read_csv(cfg["data"]["train_labels_path"])
test_df  = pd.read_csv(cfg["data"]["test_features_path"])
CLASS_NAMES = cfg["general"]["class_names"]

print(f"Train samples : {len(train_df):,}")
print(f"Test samples  : {len(test_df):,}")
print("\nClass distribution:")
print(labels[CLASS_NAMES].sum().to_string())

sample_id  = train_df["id"].iloc[0]
sample_img = LOCAL_DATA / "train_features" / f"{sample_id}.jpg"
print(f"\nSample image exists: {sample_img.exists()}  ({sample_img})")


Train samples : 16,488
Test samples  : 4,464

Class distribution:
antelope_duiker     2474.0
bird                1641.0
blank               2213.0
civet_genet         2423.0
hog                  978.0
leopard             2254.0
monkey_prosimian    2492.0
rodent              2013.0

Sample image exists: True  (/content/data_local/train_features/ZJ000000.jpg)


## 4 · Restore Checkpoints from Drive
Use this section if you want to **continue training** or **predict** in a new session
using checkpoints saved to Drive in a previous session.

In [ ]:
# ── Restore checkpoints from Drive → local ─────────────────────
import shutil
from pathlib import Path

# DRIVE_CKPT is defined in Cell 3 (Mount Drive)

local_ckpt = Path("/content/conser-vision/models/weights")
local_ckpt.mkdir(parents=True, exist_ok=True)

restored = 0
for f in DRIVE_CKPT.glob("fold*_best.pth"):
    dest = local_ckpt / f.name
    shutil.copy2(f, dest)
    print(f"  Restored {f.name}")
    restored += 1

print(f"\n✓ Restored {restored} checkpoint(s)")

  Restored fold1_best.pth
  Restored fold2_best.pth
  Restored fold3_best.pth

✓ Restored 3 checkpoint(s)


## 4 · Training


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TRAINING PARAMETERS — edit here                            ║
# ╚══════════════════════════════════════════════════════════════╝

N_FOLDS    = 5     # 5 for full CV; use 1 for a quick smoke-test
N_EPOCHS   = 30    # max epochs per fold (early stopping may stop earlier)
BATCH_SIZE = 32    # Colab T4 (~16 GB) fits batch_size=32 at 224px

# DRIVE_CKPT is defined in Cell 3 (Mount Drive)
# ╚══════════════════════════════════════════════════════════════╝

import yaml
import sys
sys.path.insert(0, "/content/conser-vision")

with open("/content/conser-vision/configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["cross_validation"]["n_splits"] = N_FOLDS
cfg["baseline"]["num_epochs"]        = N_EPOCHS
cfg["baseline"]["batch_size"]        = BATCH_SIZE

print("Config summary:")
print(f"  Model     : {cfg['baseline']['model_name']}")
print(f"  Image size: {cfg['baseline']['image_size']} px")
print(f"  Folds     : {N_FOLDS}")
print(f"  Epochs    : {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR        : {cfg['baseline']['learning_rate']}")


Config summary:
  Model     : efficientnet_b3
  Image size: 224 px
  Folds     : 5
  Epochs    : 30
  Batch size: 32
  LR        : 0.0001


In [ ]:
# ── Run cross-validation training ─────────────────────────────
import os
import sys
import torch
os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

from pathlib import Path
from utils.seed import set_global_seed
from src.data.dataset import load_dataframes
from src.training.train import run_cv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

set_global_seed(cfg["general"]["seed"])

train_df, test_df = load_dataframes(
    train_features_path=cfg["data"]["train_features_path"],
    train_labels_path=cfg["data"]["train_labels_path"],
    test_features_path=cfg["data"]["test_features_path"],
)

output_dir = Path(cfg["paths"]["model_dir"])

oof_preds, fold_scores = run_cv(
    train_df=train_df,
    images_dir=cfg["data"]["train_images_dir"],
    cfg=cfg,
    device=device,
    output_dir=output_dir,
    resume_dir=DRIVE_CKPT,
)

print("\n✓ Training complete.")
print(f"  Mean CV log-loss: {sum(fold_scores)/len(fold_scores):.4f}")


KeyboardInterrupt: 

In [ ]:
# ── Backup checkpoints to Drive ────────────────────────────────
import shutil
from pathlib import Path

DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

ckpt_dir = Path("/content/conser-vision/models/weights")
for f in ckpt_dir.glob("*.pth"):
    dest = DRIVE_CKPT / f.name
    shutil.copy2(f, dest)
    print(f"  Copied {f.name} → {dest}")

oof_src = Path("/content/conser-vision/data/processed/oof_predictions.csv")
if oof_src.exists():
    shutil.copy2(oof_src, DRIVE_CKPT / "oof_predictions.csv")
    print("  Copied oof_predictions.csv")

print("\n✓ Checkpoints backed up to Drive")


## 5 · Generate Submission


In [ ]:
# ── Generate submission CSV ────────────────────────────────────
import os
import sys
import torch
os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

from datetime import datetime
from pathlib import Path
import yaml

from src.data.dataset import load_dataframes
from src.evaluation.predict import generate_submission

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Options ───────────────────────────────────────────────────
USE_TTA  = True            # False = faster but lower quality
CKPT_DIR = "models/weights"  # or str(DRIVE_CKPT) if restoring from Drive
# ──────────────────────────────────────────────────────────────

_, test_df = load_dataframes(
    train_features_path=cfg["data"]["train_features_path"],
    train_labels_path=cfg["data"]["train_labels_path"],
    test_features_path=cfg["data"]["test_features_path"],
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tta_tag   = "TTA" if USE_TTA else "noTTA"
out_path  = f"submissions/submission_{tta_tag}_{timestamp}.csv"

sub_df = generate_submission(
    test_df=test_df,
    images_dir=cfg["data"]["test_images_dir"],
    checkpoint_dir=CKPT_DIR,
    cfg=cfg,
    device=device,
    use_tta=USE_TTA,
    output_path=out_path,
    folds=[1,2,3,4]
)


Train: 16,488 samples
Test:  4,464 samples
Class distribution:
antelope_duiker     2474.0
bird                1641.0
blank               2213.0
civet_genet         2423.0
hog                  978.0
leopard             2254.0
monkey_prosimian    2492.0
rodent              2013.0
Found 3 checkpoints: ['fold1_best.pth', 'fold2_best.pth', 'fold3_best.pth']

Loading fold1_best.pth
Loaded checkpoint from models/weights/fold1_best.pth
Model: efficientnet_b3  |  Params: 10.7M  |  Classes: 8
  TTA view 1/5


  TTA view 2/5


  TTA view 3/5


  TTA view 4/5


  TTA view 5/5


...

  TTA view 2/5


  TTA view 3/5


  TTA view 4/5


  TTA view 5/5



✓ Submission saved → submissions/submission_TTA_20260409_113049.csv
         id  antelope_duiker      bird     blank  civet_genet       hog  \
0  ZJ016488         0.175348  0.026607  0.338550     0.206281  0.013529   
1  ZJ016489         0.495860  0.145553  0.113312     0.023099  0.058643   
2  ZJ016490         0.329746  0.060274  0.176169     0.194018  0.172827   
3  ZJ016491         0.018076  0.012539  0.030833     0.032125  0.036751   
4  ZJ016492         0.151666  0.162542  0.070762     0.045936  0.055564   

    leopard  monkey_prosimian    rodent  
0  0.063453          0.055517  0.120714  
1  0.026058          0.083666  0.053811  
2  0.016784          0.028006  0.022177  
3  0.820565          0.011126  0.037986  
4  0.016105          0.396342  0.101083  


In [ ]:
# ── Copy submission to Drive + download link ───────────────────
import shutil
from pathlib import Path
from google.colab import files

DRIVE_SUB.mkdir(parents=True, exist_ok=True)

local_sub  = Path(out_path)
drive_dest = DRIVE_SUB / local_sub.name
shutil.copy2(local_sub, drive_dest)
print(f"✓ Saved to Drive: {drive_dest}")

print(f"\nStarting download of {local_sub.name} ...")
files.download(str(local_sub))


✓ Saved to Drive: /content/drive/MyDrive/ML_projects/Conser_vision/submissions/submission_TTA_20260409_113049.csv

Starting download of submission_TTA_20260409_113049.csv ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>